# Дело II-02 · Пассажиры после факта

**Бюро аналитических расследований, 7–8 апреля 2026 года.** После первого заключения Антон Карев приносит новый пример «Компаса»: прогноз выживания пассажиров «Титаника» с почти идеальной метрикой.

Тимур Сафин, хранитель данных Бюро, передаёт замороженный снимок и его контрольную сумму. Вера Орлова задаёт вопрос до запуска модели: **какие сведения существовали в момент, когда решение ещё можно было принять?** Ответ превращает обычную работу с пропусками в аудит временной доступности.

**Цель:** построить воспроизводимый конвейер предобработки и количественно показать, как `boat` и `body` превращают описание последствия в ложный «прогноз».


## Маршрут расследования

1. Проверить SHA-256 и карточку OpenML 40945.
2. Исследовать типы и пропуски.
3. Классифицировать поля по доступности в момент решения.
4. Один раз зафиксировать внешнюю тестовую выборку.
5. Собрать корректный `ColumnTransformer` с логистической регрессией.
6. Намеренно добавить утечку и сравнить результаты.
7. Разобрать ошибки и написать записку.

Ориентир — 4–5 часов. Дело автономно и не загружает данные во время анализа.


In [ ]:
from __future__ import annotations

import hashlib
import random
import urllib.request
import zipfile
from pathlib import Path

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
NOTEBOOK_VARIANT = "learner"
CASE_SLUG = "case-02"
ARCHIVE_NAME = "part-2-case-02.zip"
COURSE_SITE = "https://mkuziuk.github.io/python-tutorial"
IN_COLAB = False

# При локальном запуске используем файлы из каталога дела; в Colab скачиваем архив и проверяем его SHA-256.
try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_local_case() -> Path | None:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / f"{CASE_SLUG}.ipynb").exists()
        ):
            return candidate
        nested = candidate / "projects" / "part-2" / CASE_SLUG
        if (nested / "README.md").exists():
            return nested
    return None

def download_colab_case() -> Path:
    destination = Path("/content") / f"python-tutorial-{CASE_SLUG}"
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / ARCHIVE_NAME
    archive_url = f"{COURSE_SITE}/downloads/{ARCHIVE_NAME}"
    checksum_url = f"{archive_url}.sha256"

    urllib.request.urlretrieve(archive_url, archive_path)
    # Сверяем SHA-256 до распаковки, чтобы не выполнить код из повреждённого архива.
    with urllib.request.urlopen(checksum_url) as response:
        expected = response.read().decode("utf-8").split()[0].lower()
    actual = sha256_file(archive_path)
    if actual != expected:
        raise RuntimeError(f"SHA-256 архива не совпал: {actual} != {expected}")

    unpacked = destination / "unpacked"
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    matches = sorted(unpacked.rglob(f"{CASE_SLUG}.ipynb"))
    if not matches:
        raise FileNotFoundError(f"В архиве нет {CASE_SLUG}.ipynb")
    return matches[0].parent

# DATA_DIR и ARTIFACTS_DIR строятся от найденного каталога дела, поэтому текущая папка не влияет на пути.
CASE_DIR = find_local_case()
if CASE_DIR is None and IN_COLAB:
    CASE_DIR = download_colab_case()
if CASE_DIR is None:
    raise FileNotFoundError(
        f"Не найден каталог {CASE_SLUG}. Запустите тетрадь из каталога дела."
    )

DATA_DIR = CASE_DIR / "data"
print(f"Среда: {'Colab' if IN_COLAB else 'local'} | дело: {CASE_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Замороженный источник

В деле лежит CSV-представление OpenML dataset 40945, version 1: 1 309 пассажиров, 14 полей, цель `survived`. OpenML помечает лицензию как `Public`; подробная атрибуция находится в `data/SOURCE.md` и `data/LICENSE.txt`.

Мы проверяем локальный SHA-256 до чтения. Это не доказывает истинность исторических записей, но доказывает, что все участники курса анализируют одинаковые байты.


In [ ]:
DATASET_SHA256 = "c617db2c7470716250f6f001be51304c76bcc8815527ab8bae734bdca0735737"
data_path = DATA_DIR / "titanic.csv"
actual_sha256 = sha256_file(data_path)
if actual_sha256 != DATASET_SHA256:
    raise RuntimeError("Контрольная сумма titanic.csv не совпала")

passengers = pd.read_csv(data_path, na_values=["?"])
passengers["survived"] = passengers["survived"].astype(int)
print(f"SHA-256: {actual_sha256}")
print("Форма:", passengers.shape)
display(passengers.head(3))


## 2. Тип — ещё не смысл

`object` сообщает только способ хранения. Оно не говорит, является ли поле категорией, свободным текстом, идентификатором или информацией из будущего. Поэтому технический аудит дополняем смысловым.


In [ ]:
schema_audit = pd.DataFrame(
    {
        "dtype": passengers.dtypes.astype(str),
        "missing_n": passengers.isna().sum(),
        "missing_share": passengers.isna().mean(),
        "unique": passengers.nunique(dropna=True),
    }
).sort_values("missing_share", ascending=False)
display(schema_audit.round(3))


In [ ]:
missing_plot = schema_audit.query("missing_n > 0").sort_values("missing_share")
ax = missing_plot["missing_share"].plot.barh(figsize=(8, 4), color="#3157a4")
ax.set(xlabel="Доля пропусков", ylabel="Поле", title="Пропуски — это часть истории данных")
plt.tight_layout()
plt.show()


## 3. Упражнение: карта доступности

Момент решения фиксируем так: модель должна работать **до того, как известен исход эвакуации**. Для каждого поля назначьте один статус:

- `available` — могло быть известно в момент решения;
- `post_outcome` — появилось после исхода;
- `excluded_identifier` — идентификатор или свободный текст, который мы не используем в базовой модели;
- `target` — то, что предсказываем.

Объясните статус человеческим предложением, а не типом pandas.


In [ ]:
# TODO: замените статус и объяснение для каждого поля.
field_audit = pd.DataFrame(
    {
        "feature": passengers.columns,
        "decision_status": "TODO",
        "reason": "TODO: когда поле становится известно?",
    }
)
display(field_audit)


### Пропуск как сигнал из будущего

У `boat` значение обычно есть у спасённых, у `body` — у части погибших. Даже само наличие значения кодирует исход. Посмотрим на данные напрямую; это описательная таблица, а не причинное объяснение.


In [ ]:
leak_evidence = pd.DataFrame(
    {
        "boat_present": passengers.groupby("survived")["boat"].apply(lambda s: s.notna().mean()),
        "body_present": passengers.groupby("survived")["body"].apply(lambda s: s.notna().mean()),
        "count": passengers.groupby("survived").size(),
    }
).rename(index={0: "did_not_survive", 1: "survived"})
display(leak_evidence.round(3))


## 4. Внешняя тестовая выборка

Фиксируем индексы до предобработки. Параметры заполнения, набор категорий и коэффициенты должны определяться только по обучающей выборке. `Pipeline` обеспечивает это технически, а корректное разбиение — методологически.


In [ ]:
# Модели с утечкой и без неё используем с одинаковыми train_index и test_index.
train_index, test_index = train_test_split(
    passengers.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=passengers["survived"],
)
train_data = passengers.loc[train_index].copy()
test_data = passengers.loc[test_index].copy()
y_train = train_data["survived"]
y_test = test_data["survived"]

split_balance = pd.DataFrame(
    {"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}
).rename(index={0: "did_not_survive", 1: "survived"})
display(split_balance.round(3))
print("train:", len(train_data), "| test:", len(test_data))


## 5. Корректный конвейер

Числа: медиана + индикатор пропуска + `StandardScaler`. Категории: отдельное значение `__MISSING__` + `OneHotEncoder(handle_unknown="ignore")`. Затем логистическая регрессия.

Почему всё собрано в одном конвейере? При `.fit(X_train, y_train)` каждое обучаемое преобразование видит только train. При `.predict(X_test)` используются уже выученные параметры.


In [ ]:
# honest_features содержит только признаки, доступные в момент прогноза.
honest_numeric = ["age", "sibsp", "parch", "fare"]
honest_categorical = ["pclass", "sex", "embarked"]
honest_features = honest_numeric + honest_categorical
post_outcome_features = ["boat", "body"]

X_train_honest = train_data[honest_features]
X_test_honest = test_data[honest_features]
print("Честные признаки:", honest_features)


In [ ]:
def build_logistic_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    numeric_steps = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_steps = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessing = ColumnTransformer(
        [
            ("numeric", numeric_steps, numeric_features),
            ("categorical", categorical_steps, categorical_features),
        ]
    )
    # Импутация, кодирование и масштабирование обучаются внутри `Pipeline` только на обучающей выборке.
    return Pipeline(
        [
            ("preprocess", preprocessing),
            (
                "model",
                LogisticRegression(
                    solver="liblinear", max_iter=1000, random_state=RANDOM_STATE
                ),
            ),
        ]
    )


### Базовая модель до основной

Базовая модель `most_frequent` нужна даже при дисбалансе 62/38: accuracy около 0,62 можно получить, ни разу не предсказав выживание. Поэтому дополнительно покажем ROC AUC по вероятностям.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train_honest, y_train)
dummy_predictions = dummy.predict(X_test_honest)
dummy_accuracy = accuracy_score(y_test, dummy_predictions)
print(f"Dummy accuracy: {dummy_accuracy:.3f}")


In [ ]:
honest_model = build_logistic_pipeline(honest_numeric, honest_categorical)
honest_model.fit(X_train_honest, y_train)
honest_predictions = honest_model.predict(X_test_honest)
honest_probabilities = honest_model.predict_proba(X_test_honest)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_predictions)
honest_auc = roc_auc_score(y_test, honest_probabilities)
print(f"Корректная модель | доля правильных ответов={honest_accuracy:.3f} | ROC AUC={honest_auc:.3f}")


## 6. Контролируемый эксперимент с утечкой

Теперь намеренно добавим `boat` как категорию и `body` как число. Это **не кандидат для внедрения**, а отрицательный контроль: он показывает размер и направление ошибки постановки.

Отдельная категория `__MISSING__` особенно важна: отсутствие номера лодки или тела само связано с тем, что произошло после катастрофы.


In [ ]:
leaky_numeric = honest_numeric + ["body"]
leaky_categorical = honest_categorical + ["boat"]
leaky_features = leaky_numeric + leaky_categorical

# Этот вариант намеренно нарушает временную границу и нужен только для измерения утечки.
leaky_model = build_logistic_pipeline(leaky_numeric, leaky_categorical)
leaky_model.fit(train_data[leaky_features], y_train)
leaky_predictions = leaky_model.predict(test_data[leaky_features])
leaky_probabilities = leaky_model.predict_proba(test_data[leaky_features])[:, 1]

leaky_accuracy = accuracy_score(y_test, leaky_predictions)
leaky_auc = roc_auc_score(y_test, leaky_probabilities)
comparison = pd.DataFrame(
    [
        {"scenario": "dummy", "accuracy": dummy_accuracy, "roc_auc": 0.5, "deployable": False},
        {"scenario": "honest", "accuracy": honest_accuracy, "roc_auc": honest_auc, "deployable": "needs validation"},
        {"scenario": "post-outcome leak", "accuracy": leaky_accuracy, "roc_auc": leaky_auc, "deployable": False},
    ]
)
display(comparison.round(3))


**Упражнение.** Сформулируйте механизм скачка без слов «модель умнее». Свяжите момент появления поля, пропуски и цель. Затем укажите, почему случайное разбиение на обучающую и тестовую выборки само по себе не защищает от такой утечки.


In [ ]:
# TODO: объясните скачок через момент доступности, а не через алгоритм.
leak_explanation = "TODO"
print(leak_explanation)


## 7. Ошибки корректной модели

Таблица нужна не для поиска «виноватых пассажиров», а для проверки систематических режимов ошибки. На этом этапе только описываем; пороги и групповые метрики станут центральной темой II-03.


In [ ]:
honest_error_table = test_data[["pclass", "sex", "age", "fare", "survived"]].copy()
honest_error_table["predicted"] = honest_predictions
honest_error_table["probability"] = honest_probabilities
honest_error_table["error"] = honest_error_table["survived"] != honest_error_table["predicted"]

error_summary = (
    honest_error_table.groupby(["sex", "pclass"], dropna=False)
    .agg(n=("error", "size"), errors=("error", "sum"), error_rate=("error", "mean"))
    .sort_values("error_rate", ascending=False)
)
display(error_summary.round(3))


> **Типичная ошибка:** удалить столбец только потому, что у него много пропусков, или оставить его только потому, что конвейер умеет заполнять пропуски. Сначала выясняем происхождение и время появления поля; потом выбираем техническую обработку.

**Дополнительное исследование:** сравните распределения `fare` в обучающей и тестовой выборках. Это полезная диагностика, но не используйте тестовую выборку для подбора правил обработки.


## 8. Аудиторская записка

Установленный факт должен ссылаться на конкретные поля и сравнение. Интерпретация может говорить о несостоятельности заявления, которое поставщик подкрепляет контрольной оценкой. Утверждение о личном умысле в данных отсутствует.


In [ ]:
audit_memo = {
    "established_fact": "TODO",
    "supported_interpretation": "TODO",
    "not_proven": "TODO",
    "limitations": "TODO",
    "recommended_action": "TODO",
}
display(pd.Series(audit_memo, name="Записка II-02").to_frame())


In [ ]:
fitted_numeric_pipeline = honest_model.named_steps["preprocess"].named_transformers_[
    "numeric"
]
fitted_numeric_imputer = fitted_numeric_pipeline.named_steps["imputer"]
fitted_numeric_scaler = fitted_numeric_pipeline.named_steps["scaler"]

expected_train_medians = X_train_honest[honest_numeric].median().to_numpy()
imputed_train_numeric = fitted_numeric_imputer.transform(
    X_train_honest[honest_numeric]
)
imputed_all_numeric = fitted_numeric_imputer.transform(
    passengers[honest_numeric]
)
numeric_preprocessing_fit_on_train_only = (
    np.allclose(fitted_numeric_imputer.statistics_, expected_train_medians)
    and np.allclose(
        fitted_numeric_scaler.mean_, imputed_train_numeric.mean(axis=0)
    )
    and not np.allclose(
        fitted_numeric_scaler.mean_, imputed_all_numeric.mean(axis=0)
    )
)

checks = {
    "snapshot_shape": passengers.shape == (1309, 14),
    "split_disjoint": set(train_index).isdisjoint(test_index),
    "honest_fields_exclude_future": set(post_outcome_features).isdisjoint(honest_features),
    "pipeline_beats_dummy": honest_accuracy > dummy_accuracy,
    "honest_auc_in_broad_range": 0.80 < honest_auc < 0.92,
    "leaky_auc_above_098": 0.98 < leaky_auc <= 1.0,
    "leak_creates_large_jump": leaky_auc - honest_auc > 0.10,
    "numeric_preprocessing_train_only": numeric_preprocessing_fit_on_train_only,
    "checksum_verified": actual_sha256 == DATASET_SHA256,
}
display(pd.Series(checks, name="passed").to_frame())
if NOTEBOOK_VARIANT == "solution":
    assert all(checks.values())
    assert np.allclose(
        fitted_numeric_imputer.statistics_,
        X_train_honest[honest_numeric].median().to_numpy(),
    )
    assert np.allclose(
        fitted_numeric_scaler.mean_, imputed_train_numeric.mean(axis=0)
    )
    assert not np.allclose(
        fitted_numeric_scaler.mean_, imputed_all_numeric.mean(axis=0)
    )
    statuses = field_audit.set_index("feature")["decision_status"]
    assert statuses["boat"] == statuses["body"] == "post_outcome"
else:
    print("Строгая проверка карты доступности включится после TODO.")


## Дело закрыто

Перезапустите ядро, выполните **Run All** и сравните инварианты с `check_result.md`. В II-03 Бюро сохранит этот корректный конвейер, но спросит: одинаково ли полезны все правильные ответы и кому достаются ошибки?
